# Mill Street Labs · local CPU sandbox

Run this notebook with **Run → Run All Cells**, or step through it with **Shift+Enter**.

1. Watch an observation-limited heuristic population in the game.
2. Set a small CPU budget and train actual **SmolLM2 + private LoRA/GRU policies**.
3. Inspect population dynamics, policy quality, behavior probes, and saved checkpoints.

The default run is deliberately short: roughly **1–2 minutes with cached weights**, depending on your CPU. Set `BACKEND = "tiny"` for a seconds-long functional check. First-time model downloads add time. This is an execution demo, not evidence of convergence or inherited improvement.

Launch from the repository: `python -m jupyterlab notebooks/local_sandbox.ipynb`.


In [ ]:
from pathlib import Path
import os
import sys

# Works when Jupyter starts in either the repository or notebooks/ directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/millstlabs").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the millstlabs repository.")

# Reuse weights already downloaded by the CLI; keep plotting caches local.
os.environ.setdefault("HF_HOME", str(ROOT / ".cache/huggingface"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
sys.path.insert(0, str(ROOT / "src"))

import copy
import gc
import json
import time
from datetime import datetime
from uuid import uuid4

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap
from IPython.display import HTML, FileLink, display

from millstlabs.config import load_config
from millstlabs.env import ACTION_NAMES, PopulationEnv
from millstlabs.observations import Heuristic

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": False})
print(f"Python: {sys.executable}")
print(f"Repository: {ROOT}")


## 1. Watch the sandbox

This animation uses **heuristics**, not the model you train below. It is a quick way to inspect feeding, predation, and births. The full-world rendering is for you only; policies receive local observations.

**Blue** = prey, **teal** = alert prey, **orange** = food, **red** = predator, **dark gray** = obstacles. A predator can overlap a prey; the red cell takes visual precedence.


In [ ]:
DEMO_TICKS = 128
DEMO_SEED = 11

def world_image(env):
    grid = np.zeros((env.cfg.size, env.cfg.size), dtype=np.int8)
    for x, y in env.obstacles:
        grid[y, x] = 1
    for x, y in env.station_positions:
        grid[y, x] = 2
    for body in env.prey.values():
        x, y = body.pos
        grid[y, x] = 4 if body.alert_until >= env.tick else 3
    x, y = env.predator
    grid[y, x] = 5
    return grid

demo_cfg = load_config(ROOT / "configs/smoke.yaml")
demo = PopulationEnv(demo_cfg.environment, reproduction=True)
observations, _ = demo.reset(seed=DEMO_SEED)
heuristics = {}
frames = [(world_image(demo), demo.metrics())]
for _ in range(DEMO_TICKS):
    if not demo.agents:
        break
    for agent in demo.agents:
        if agent not in heuristics:
            heuristics[agent] = Heuristic(DEMO_SEED + int(agent.split("_")[-1]))
    observations, *_ = demo.step({agent: heuristics[agent].act(observations[agent])
                                  for agent in demo.agents})
    if demo.tick % 4 == 0 or not demo.agents:
        frames.append((world_image(demo), demo.metrics()))

colors = ListedColormap(["#f3f4f6", "#374151", "#f59e0b", "#2563eb", "#14b8a6", "#ef4444"])
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(frames[0][0], cmap=colors, vmin=0, vmax=5, interpolation="nearest")
ax.set_xticks([])
ax.set_yticks([])
def animate(index):
    grid, metrics = frames[index]
    im.set_data(grid)
    ax.set_title(f"Tick {metrics['tick']} · {metrics['population']} prey · {metrics['births']} births")
    return (im,)

animation = FuncAnimation(fig, animate, frames=len(frames), interval=160, blit=False)
html_animation = animation.to_jshtml(default_mode="once")
plt.close(fig)
display(HTML(html_animation))
print({key: demo.metrics()[key] for key in ["tick", "population", "births", "matured", "starvation", "predation"]})


## 2. Choose your experiment

Edit the controls below, then rerun from this cell. Each execution creates a fresh output directory, so it does not overwrite an earlier experiment.

- `BACKEND`: `"smollm"` is the actual experiment architecture; `"tiny"` is a functional test.
- `METHODS`: start with `["r_adult"]`; use `["iteration", "r_adult", "r_initial"]` to compare all three with one shared warm start. That takes roughly three times as long.
- `TRAIN_DECISIONS` counts individual actions, **not** world ticks. Eight living prey consume eight decisions per tick.
- `EVALUATION_HORIZON` controls extra evaluation work. Keep it short for a local demo.

The population uses 8 founders, a cap of 16, and a fixed social denominator of 16. Small runs often have no births and evaluations can hit the short horizon; neither result establishes an inheritance advantage.


In [ ]:
BACKEND = "smollm"
PROFILE = "prosocial"
METHODS = ["r_adult"]
SEED = 11
TRAIN_DECISIONS = 128
WARMSTART_TRANSITIONS = 16
EVALUATION_HORIZON = 32

cfg = load_config(ROOT / "configs/model-check.yaml")
cfg.training.backend = BACKEND
cfg.training.device = "cpu"
cfg.training.decisions = TRAIN_DECISIONS
cfg.training.warmstart_transitions = WARMSTART_TRANSITIONS
cfg.training.evaluation_horizon = EVALUATION_HORIZON
cfg.training.checkpoints = [TRAIN_DECISIONS]
cfg.validate()
assert PROFILE in cfg.profiles
assert METHODS and len(set(METHODS)) == len(METHODS)
assert all(method in cfg.methods for method in METHODS)

run_id = datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
RUN_ROOT = ROOT / "runs/notebook" / run_id
RUN_ROOT.mkdir(parents=True)
run_paths = {}
print(f"Backend: {BACKEND} | CPU | {TRAIN_DECISIONS} decisions per method")
print(f"Methods: {', '.join(METHODS)} | profile: {PROFILE} | seed: {SEED}")
print(f"Output: {RUN_ROOT}")


## 3. Common warm start

All selected methods use this exact checkpoint. The full experiment calls for 50,000 demonstration transitions; this demo uses only the small number selected above. If you rerun this cell, it reuses a checkpoint only when its config and seed match.


In [ ]:
import torch
from millstlabs.trainer import Trainer, warmstart
from millstlabs.experiments import analyze

warm_path = RUN_ROOT / "warm.pt"
started = time.perf_counter()
if warm_path.exists():
    warm = torch.load(warm_path, map_location="cpu", weights_only=False)
    if warm["config_digest"] != cfg.digest() or warm["seed"] != SEED:
        raise ValueError("Warm-start settings changed. Rerun the settings cell for a fresh run.")
    print("Reusing the matching warm start.")
else:
    warm = warmstart(cfg, SEED, warm_path)
print(f"Warm start: {warm['transitions']} transitions in {time.perf_counter() - started:.1f}s")
gc.collect()


## 4. Train and evaluate

This calls `Trainer.run()`, the same runner as the command line. Progress appears after rollouts; the final evaluation and probes can take additional time. It writes resume snapshots as it goes.

To interrupt, use Jupyter's stop button. Rerun this cell to resume the most recent committed checkpoint. If interrupted before the first checkpoint, the partial output is kept and a new attempt directory is used. A completed run is displayed without retraining when this cell is rerun.


In [ ]:
summaries = {}
for method in METHODS:
    output = run_paths.get(method, RUN_ROOT / method)
    checkpoint = output / "latest.pt"
    summary_path = output / "summary.json"
    if summary_path.exists() and json.loads(summary_path.read_text()).get("finished"):
        summaries[method] = json.loads(summary_path.read_text())
        run_paths[method] = output
        print(f"{method}: already complete, reusing outputs")
        continue
    if output.exists() and any(output.iterdir()) and not checkpoint.exists():
        output = RUN_ROOT / f"{method}-retry-{uuid4().hex[:6]}"
        checkpoint = output / "latest.pt"
    run_paths[method] = output
    print(f"\nRunning {method} …", flush=True)
    runner = Trainer(cfg, PROFILE, method, SEED, output, warm_path,
                     resume=checkpoint if checkpoint.exists() else None)
    try:
        summaries[method] = runner.run()
        print(f"{method}: {summaries[method]['decisions']} decisions, "
              f"{summaries[method]['world_ticks']} world ticks, "
              f"{summaries[method]['elapsed_seconds']:.1f}s", flush=True)
    finally:
        del runner
        gc.collect()

summary_rows = [{"method": method, **{k: summary[k] for k in
                ["decisions", "world_ticks", "elapsed_seconds", "evaluation_decisions", "births", "restarts"]}}
                for method, summary in summaries.items()]
display(summary_rows)


## 5. Inspect population dynamics

These plots describe **native training ecology**. World tick continues across external restarts; cumulative food and death counters are reconciled across ecological trials. Action shares aggregate the seven actions selected by living prey.


In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.exists() else []

def cumulative_over_trials(rows, key):
    total, previous_trial, previous_value, values = 0, None, 0, []
    for row in rows:
        if previous_trial is not None and row["trial"] != previous_trial:
            total += previous_value
        values.append(total + row[key])
        previous_trial, previous_value = row["trial"], row[key]
    return values

fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
for method, output in run_paths.items():
    rows = read_jsonl(output / "ecology.jsonl")
    if not rows:
        continue
    ticks = [r["world_ticks"] for r in rows]
    axes[0, 0].plot(ticks, [r["population"] for r in rows], label=method)
    axes[0, 1].plot(ticks, cumulative_over_trials(rows, "consumption"), label=method)
    for cause, style in [("starvation", "-"), ("predation", "--")]:
        axes[1, 0].plot(ticks, cumulative_over_trials(rows, cause), style, label=f"{method}: {cause}")
    actions = np.array([sum(r["actions"].get(str(a), 0) for r in rows) for a in range(7)])
    axes[1, 1].plot(ACTION_NAMES, actions / max(actions.sum(), 1), marker="o", label=method)

axes[0, 0].axhline(cfg.environment.population_cap, color="gray", linestyle=":", label="population cap")
axes[0, 0].set(title="Living population", xlabel="World tick", ylabel="Prey")
axes[0, 1].set(title="Food consumed", xlabel="World tick", ylabel="Energy units")
axes[1, 0].set(title="Causes of death", xlabel="World tick", ylabel="Cumulative deaths")
axes[1, 1].set(title="Training behavior", ylabel="Share of decisions")
axes[1, 1].tick_params(axis="x", rotation=35)
for ax in axes.flat:
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8)
fig.savefig(RUN_ROOT / "ecology.png", bbox_inches="tight")
plt.show()


## 6. Held-out policy quality

Reproduction and learning are disabled during evaluation. Bodies and recurrent memories reset. Each bar is one trained run's mean over its evaluation maps; **one training seed is not enough to estimate uncertainty or establish a winning method**. The dashed line is the short evaluation horizon, not a target score.


In [ ]:
evaluation_rows = []
for method, output in run_paths.items():
    rows = read_jsonl(output / "evaluation.jsonl")
    if rows:
        last = rows[-1]
        evaluation_rows.append({"method": method,
                                "restricted_mean_lifetime": last["restricted_mean_lifetime"],
                                "survival_fraction": last["survival_fraction"],
                                "sampled_population": last["population"]})
display(evaluation_rows)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([r["method"] for r in evaluation_rows], [r["restricted_mean_lifetime"] for r in evaluation_rows], color="#2563eb")
ax.axhline(EVALUATION_HORIZON, color="gray", linestyle="--", label="evaluation horizon")
ax.set(ylabel="Mean restricted lifetime (ticks)", title="Frozen policies on held-out maps")
ax.legend()
fig.tight_layout()
fig.savefig(RUN_ROOT / "evaluation.png", bbox_inches="tight")
plt.show()


## 7. Behavioral probes

Each policy replays the same observation history using its **own recurrent state**. These are controlled sensory scenarios. The heatmap reports action probabilities, not realized survival benefits.


In [ ]:
fig, axes = plt.subplots(1, len(run_paths), figsize=(8 * len(run_paths), 4), squeeze=False)
for ax, (method, output) in zip(axes.flat, run_paths.items()):
    rows = read_jsonl(output / "probes.jsonl")
    if not rows:
        ax.set_visible(False)
        continue
    probabilities = rows[-1]["probabilities"]
    names = list(probabilities)
    values = np.array([probabilities[name] for name in names])
    image = ax.imshow(values, vmin=0, vmax=1, cmap="Blues", aspect="auto")
    ax.set_xticks(range(7), ACTION_NAMES, rotation=35)
    ax.set_yticks(range(len(names)), [name.replace("_", " ") for name in names])
    for y in range(len(names)):
        for x in range(7):
            ax.text(x, y, f"{values[y, x]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if values[y, x] > 0.55 else "black")
    ax.set_title(method)
    fig.colorbar(image, ax=ax, label="Action probability")
fig.tight_layout()
fig.savefig(RUN_ROOT / "probes.png", bbox_inches="tight")
plt.show()


## 8. Saved results and next runs

`analysis.json` aggregates by training seed and includes paired method differences when the selected methods permit them. With this single seed, uncertainty intervals remain empty. Additional files include per-agent PPO updates, birth/death/alarm events, resource usage, and full recovery checkpoints.

To run longer, increase `TRAIN_DECISIONS` and optionally the evaluation horizon in the settings cell, then run from there. Keep CPU costs in mind: the earlier model benchmark measured roughly 7.8 decisions/second for collection plus PPO, before evaluation/probes. More useful comparisons require longer learning and multiple independent seeds.

The full experiment configuration and methodology remain in `configs/main.yaml`, `docs/experiment.md`, and `docs/compute.md`; changing notebook controls does not modify them.


In [ ]:
# Avoid including incomplete retries in the paired analysis: collect only the
# selected completed run directories into a temporary view using file copies.
import shutil
analysis_inputs = RUN_ROOT / "analysis-inputs"
if analysis_inputs.exists():
    shutil.rmtree(analysis_inputs)
for method, output in run_paths.items():
    target = analysis_inputs / method
    target.mkdir(parents=True)
    for path in output.glob("*.jsonl"):
        shutil.copy2(path, target / path.name)
analysis = analyze(analysis_inputs, RUN_ROOT / "analysis.json")
shutil.rmtree(analysis_inputs)
print(f"Saved to {RUN_ROOT}")
print("Paired method comparisons:", analysis["paired_contrasts"])
# Links are relative to the notebook directory for JupyterLab's file handler.
for filename in ["analysis.json", "ecology.png", "evaluation.png", "probes.png"]:
    relative = os.path.relpath(RUN_ROOT / filename, ROOT / "notebooks")
    display(FileLink(relative))
print("Per-method metrics and checkpoints:")
for method, output in run_paths.items():
    print(f"  {method}: {output}")
